# Polaris/S3 and Hive/HDFS Iceberg on OpenShift

Follow README.md terminal preparation first. Use the existing Jupyter container and a fresh Python kernel. Run cells in order. This notebook uses in-cluster Service names, not VM/Tailscale IPs. No pip installs or image builds are needed on the tested Spark image.

The supplied original used `olist`; this example reuses the lab’s existing Polaris catalog `training` and bucket `warehouse`. It creates a unique namespace and keeps its test artifacts. See README for the catalog distinction and validation limits.

## Configuration and standalone Spark session

In [ ]:
import datetime
import json
import os
import re
import socket
import uuid
from pyspark.sql import SparkSession

RUN_ID = os.environ.get('COURSE06_RUN_ID') or ('course06_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
if not re.fullmatch(r'[a-z][a-z0-9_]{0,90}', RUN_ID):
    raise ValueError('COURSE06_RUN_ID must be a simple lowercase SQL identifier')
HDFS_BASE = os.environ.get('COURSE06_HDFS_BASE', '/training/06-polaris-iceberg').rstrip('/')
if not HDFS_BASE.startswith('/training/') or '..' in HDFS_BASE.split('/'):
    raise ValueError('Use a dedicated absolute directory under /training')
HDFS_ROOT = 'hdfs://namenode:9000' + HDFS_BASE + '/' + RUN_ID
S3_ENDPOINT = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
POLARIS_URI = os.environ.get('POLARIS_URI', 'http://polaris:8181/api/catalog').rstrip('/')
POLARIS_CATALOG = os.environ.get('COURSE06_POLARIS_CATALOG', 'training')
ACCESS = os.environ.get('AWS_ACCESS_KEY_ID', 'team')
SECRET = os.environ.get('AWS_SECRET_ACCESS_KEY', 'team1234')
CLIENT = os.environ.get('POLARIS_CLIENT_ID', 'team')
CLIENT_SECRET = os.environ.get('POLARIS_CLIENT_SECRET', 'team1234')
driver_ip = os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname())

builder = (SparkSession.builder.appName('course06-polaris-hive-' + RUN_ID)
    .master('spark://spark-master:7077')
    .config('spark.driver.host', driver_ip)
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.cores.max', '1').config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m').config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type', 'hive')
    .config('spark.sql.catalog.iceberg.uri', 'thrift://hive-metastore:9083')
    .config('spark.sql.catalog.iceberg.warehouse', HDFS_ROOT + '/warehouse')
    .config('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', ACCESS)
    .config('spark.hadoop.fs.s3a.secret.key', SECRET)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.ipc.client.connect.timeout', '10000')
    .config('spark.hadoop.ipc.client.connect.max.retries', '1')
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', POLARIS_URI)
    .config('spark.sql.catalog.polaris.warehouse', POLARIS_CATALOG)
    .config('spark.sql.catalog.polaris.oauth2-server-uri', POLARIS_URI + '/v1/oauth/tokens')
    .config('spark.sql.catalog.polaris.credential', CLIENT + ':' + CLIENT_SECRET)
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', S3_ENDPOINT)
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', ACCESS)
    .config('spark.sql.catalog.polaris.s3.secret-access-key', SECRET)
    .config('spark.sql.catalog.polaris.client.region', 'us-east-1')
    .config('spark.redaction.regex', '(?i)secret|password|token|access[.]?key|credential'))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('RUN_ID:', RUN_ID)
print('Spark:', spark.version, 'master:', spark.sparkContext.master)
print('HDFS run directory:', HDFS_ROOT)
print('Polaris warehouse/catalog:', POLARIS_CATALOG)


## Common table definition and verification

In [ ]:
SCHEMA = '(customer_id BIGINT NOT NULL, name STRING, city STRING, amount DECIMAL(10,2))'
PROPERTIES = """TBLPROPERTIES (
 'format-version'='2', 'write.format.default'='parquet',
 'write.parquet.compression-codec'='zstd', 'write.update.mode'='copy-on-write',
 'write.delete.mode'='copy-on-write', 'write.merge.mode'='copy-on-write')"""
VALUES = """(1, 'Alice', 'Bangalore', CAST(1200 AS DECIMAL(10,2))),
 (2, 'Bob', 'Hyderabad', CAST(1800 AS DECIMAL(10,2))),
 (3, 'Carol', 'Chennai', CAST(950 AS DECIMAL(10,2)))"""


def fill_and_check(table):
    # MERGE makes rerunning a cell with this same RUN_ID safe.
    spark.sql(f"""MERGE INTO {table} t USING (SELECT * FROM VALUES {VALUES}
      AS v(customer_id, name, city, amount)) s ON t.customer_id = s.customer_id
      WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    rows = spark.table(table).orderBy('customer_id').collect()
    assert [(r.customer_id, r.name, r.city, str(r.amount)) for r in rows] == [
        (1, 'Alice', 'Bangalore', '1200.00'), (2, 'Bob', 'Hyderabad', '1800.00'),
        (3, 'Carol', 'Chennai', '950.00')], rows
    spark.table(table).orderBy('customer_id').show(truncate=False)
    files = spark.sql(f'SELECT file_path FROM {table}.files').collect()
    assert files, 'No data files recorded in Iceberg metadata'
    print('TABLE:', table)
    print('DATA_FILE:', files[0].file_path)
    return files


## SeaweedFS direct S3 write/read

In [ ]:
def seaweedfs_exercise():
    path = 's3a://warehouse/course06/' + RUN_ID + '/parquet'
    # Overwrite only this run's own test path if rerunning the cell.
    spark.range(3).write.mode('overwrite').parquet(path)
    assert spark.read.parquet(path).count() == 3
    print('S3_WRITE_READ_PASS:', path)


In [ ]:
seaweedfs_exercise()


## Polaris catalog and Iceberg files in SeaweedFS

In [ ]:
def polaris_exercise():
    spark.sql('SHOW NAMESPACES IN polaris').show(truncate=False)
    spark.sql(f'CREATE NAMESPACE IF NOT EXISTS polaris.{RUN_ID}')
    table = f'polaris.{RUN_ID}.s3_polaris_customers'
    spark.sql(f'CREATE TABLE IF NOT EXISTS {table} {SCHEMA} USING iceberg {PROPERTIES}')
    files = fill_and_check(table)
    assert all(r.file_path.startswith(('s3://', 's3a://')) for r in files)
    spark.sql(f'SELECT snapshot_id, operation FROM {table}.snapshots').show(truncate=False)
    print('POLARIS_S3_ICEBERG_PASS')


In [ ]:
polaris_exercise()


## NameNode/DataNode check and direct HDFS read/write

In [ ]:
def hdfs_exercise():
    jvm = spark.sparkContext._jvm
    conf = spark.sparkContext._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI('hdfs://namenode:9000'), conf)
    action = jvm.org.apache.hadoop.hdfs.protocol.HdfsConstants.SafeModeAction.SAFEMODE_GET
    if fs.setSafeMode(action) or fs.getStatus().getCapacity() == 0:
        raise RuntimeError('HDFS blocked: safe mode or no registered DataNode. Fix registration first; do not force safe mode off.')
    path = HDFS_ROOT + '/parquet'
    spark.range(3).write.mode('overwrite').parquet(path)
    assert spark.read.parquet(path).count() == 3
    print('HDFS_WRITE_READ_PASS:', path)


The next Hive/HDFS exercise invokes this preflight before writing. If HDFS is in safe mode or has no registered DataNode, it stops with a clear error. Polaris/S3 results above remain independent.

## Iceberg registered in Hive Metastore, data stored on HDFS

In [ ]:
def hive_hdfs_exercise():
    hdfs_exercise()
    namespace = f'iceberg.{RUN_ID}'
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {namespace} LOCATION '{HDFS_ROOT}/warehouse'")
    table = namespace + '.hdfs_hive_customers'
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} {SCHEMA} USING iceberg LOCATION '{HDFS_ROOT}/customers' {PROPERTIES}")
    files = fill_and_check(table)
    assert all(r.file_path.startswith('hdfs://') for r in files)
    print('HIVE_HDFS_ICEBERG_PASS')


In [ ]:
hive_hdfs_exercise()


## Finish
The run namespace, tables and storage files are retained. Restarting this kernel creates a new RUN_ID; rerunning a table cell with the existing RUN_ID merges the same three sample rows rather than duplicating them. Stop this session when finished so the one-worker lab is available to other exercises.

In [ ]:
spark.stop()
